In [ ]:
# ============================================================================
# ETAPA 6b — GeoFM v3 REVISADO: ATENÇÃO ESPACIAL CORRIGIDA
# ============================================================================
# Diagnóstico etapa6 (Caminho C):
#   - Atenção espacial e temporal colapsaram para uniforme (todos ~0.2000)
#   - Causa: dependência circular no SpatialAttentionModule
#     O mesmo spatial_encoder era usado para computar a query E o output,
#     diluindo o gradiente até os pesos de atenção.
#   - A melhoria de +1.8pp vinha do encoding por frame, não da atenção.
#
# Correções implementadas (Caminho B):
#   1. Cross-attention com query aprendida (parâmetro, não computada)
#      → Remove a circularidade: query é estável, keys e values são
#        projeções independentes do input
#   2. Networks separados para keys e values
#      → key_proj:  posição → chave (para scoring)
#      → val_proj:  posição → valor (para output)
#   3. Regularização de entropia na loss
#      → Penaliza distribuições uniformes, força discriminação de posições
#
# Referências (mesmo split hexagonal):
#   Baseline v2:  61.3% acc, F1=0.593
#   Multi-Head v2: 60.3% acc, F1=0.578
#   GeoFM v3 (etapa6, atenção colapsada): 63.1% acc, F1=0.596
# ============================================================================

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import json
from datetime import datetime
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Referências
REFS = {
    'Baseline v2':   {'acc': 0.6126, 'f1': 0.5926, 'auc': 0.6547, 'gap': -0.0374},
    'Multi-Head v2': {'acc': 0.6027, 'f1': 0.5777, 'auc': 0.6482, 'gap': -0.0432},
    'GeoFM v3 (attn colapsada)': {'acc': 0.6306, 'f1': 0.5959, 'auc': 0.6660, 'gap': -0.0274},
}

print(f"✅ Configuração OK!")
print(f"   Device:   {DEVICE}")
print(f"   PyTorch:  {torch.__version__}")
print(f"   Notebook: ETAPA 6b — GeoFM v3 Revisado")
print()
print(f"📊 Referências (mesmo split espacial):")
for name, r in REFS.items():
    print(f"   {name:<35} acc={r['acc']:.1%}  F1={r['f1']:.3f}")

In [ ]:
# ============================================================================
# MÓDULO DE ATENÇÃO REVISADO — Cross-Attention com Query Aprendida
# ============================================================================
#
# Funcionamento:
#
#   Para cada ano t, dado o grid 7×7 (49 posições):
#
#   1. Embedding de posição:
#      Cada posição i recebe seu valor LULC e um embedding de posição 2D.
#      pos_embed: (49, embed_dim) — aprendido, captura vizinhança relativa
#
#   2. Projeções independentes:
#      key_i  = key_proj(x_i + pos_embed_i)   → para calcular score
#      val_i  = val_proj(x_i + pos_embed_i)   → para calcular output
#
#   3. Query aprendida (parâmetro fixo do modelo):
#      q = learned_query                       → "o que procuro no patch"
#
#   4. Cross-attention:
#      score_i = dot(q, key_i) / sqrt(embed_dim)
#      attn_w  = softmax(scores)               → (49,) por amostra
#      context = sum(attn_w_i * val_i)         → (context_dim,)
#
#   5. Atenção temporal sobre os 5 contextos:
#      learned_temp_query → cross-attention sobre 5 contextos
#
# Regularização de entropia:
#   H(attn) = -sum(attn_w * log(attn_w + eps))
#   Loss total = BCE + lambda_entropy * (-H)   ← penaliza alta entropia
#   (alta entropia = distribuição uniforme; penalizar força discriminação)
# ============================================================================

class SpatialAttentionV2(nn.Module):
    """
    Módulo de atenção espacial corrigido.
    Cross-attention com query aprendida — sem dependência circular.

    Input:  patch (batch, patch_years, patch_n, patch_n)
    Output: context (batch, out_dim),
            spatial_weights (batch, patch_years, n_positions),
            temporal_weights (batch, patch_years)
    """

    def __init__(self, patch_years=5, patch_n=7,
                 embed_dim=32, context_dim=64,
                 out_dim=64, dropout=0.3):
        super().__init__()

        self.patch_years = patch_years
        self.patch_n     = patch_n
        self.n_pos       = patch_n * patch_n   # 49
        self.embed_dim   = embed_dim
        self.context_dim = context_dim

        # ── Embedding de posição 2D (linha × coluna) ──────────────────────
        # Aprendido: captura relações espaciais relativas ao pixel central
        self.pos_embed = nn.Embedding(self.n_pos, embed_dim)
        # Inicializar com posições relativas ao centro [3,3]
        with torch.no_grad():
            for i in range(patch_n):
                for j in range(patch_n):
                    idx = i * patch_n + j
                    # Distância ao centro como prior inicial
                    dist = ((i - patch_n//2)**2 + (j - patch_n//2)**2) ** 0.5
                    self.pos_embed.weight[idx].fill_(0.0)
                    if embed_dim >= 2:
                        self.pos_embed.weight[idx, 0] = (i - patch_n//2) / patch_n
                        self.pos_embed.weight[idx, 1] = (j - patch_n//2) / patch_n

        # ── Projeção do valor LULC para embedding ─────────────────────────
        # Input: valor escalar LULC normalizado → embedding
        self.lulc_embed = nn.Linear(1, embed_dim)

        # ── Keys e Values — SEPARADOS ────────────────────────────────────
        # Nenhum deles é usado para computar o outro (sem circularidade)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.val_proj = nn.Linear(embed_dim, context_dim)

        # ── Query APRENDIDA para atenção espacial ─────────────────────────
        # Parâmetro fixo: o modelo aprende "o que buscar no patch"
        # Não depende do input → sem circularidade
        self.spatial_query = nn.Parameter(torch.randn(1, embed_dim) * 0.1)

        # ── Query APRENDIDA para atenção temporal ─────────────────────────
        self.temporal_query = nn.Parameter(torch.randn(1, context_dim) * 0.1)
        self.temp_key_proj  = nn.Linear(context_dim, context_dim)
        self.temp_val_proj  = nn.Linear(context_dim, context_dim)

        # ── Projeção de saída ─────────────────────────────────────────────
        self.output_proj = nn.Sequential(
            nn.Linear(context_dim, out_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.scale_spatial  = embed_dim  ** 0.5
        self.scale_temporal = context_dim ** 0.5

    def forward(self, patch):
        B = patch.shape[0]

        # Normalizar LULC (classes 0-27)
        x = patch / 27.0   # (B, 5, 7, 7)
        x_flat = x.view(B, self.patch_years, self.n_pos)  # (B, 5, 49)

        # Índices de posição (fixos)
        pos_idx = torch.arange(self.n_pos, device=patch.device)  # (49,)
        pos_emb = self.pos_embed(pos_idx)   # (49, embed_dim)

        spatial_contexts = []
        spatial_weights_list = []

        for t in range(self.patch_years):
            x_t = x_flat[:, t, :]   # (B, 49) — valores LULC do ano t

            # Embedding LULC: (B, 49, embed_dim)
            lulc_emb = self.lulc_embed(
                x_t.unsqueeze(-1))   # (B, 49, 1) → (B, 49, embed_dim)

            # Combinar embedding LULC + embedding de posição
            # pos_emb: (49, embed_dim) → broadcast para (B, 49, embed_dim)
            token_emb = lulc_emb + pos_emb.unsqueeze(0)  # (B, 49, embed_dim)

            # Keys e Values independentes
            keys   = self.key_proj(token_emb)   # (B, 49, embed_dim)
            values = self.val_proj(token_emb)   # (B, 49, context_dim)

            # Cross-attention: query aprendida vs keys
            # query: (1, embed_dim) → (B, 1, embed_dim)
            q = self.spatial_query.unsqueeze(0).expand(B, -1, -1)  # (B,1,embed_dim)

            # Scores: (B, 1, embed_dim) × (B, embed_dim, 49) → (B, 1, 49)
            scores = torch.bmm(q, keys.transpose(1, 2)) / self.scale_spatial
            attn_w = F.softmax(scores, dim=-1)   # (B, 1, 49)

            # Contexto: (B, 1, 49) × (B, 49, context_dim) → (B, 1, context_dim)
            ctx_t = torch.bmm(attn_w, values).squeeze(1)  # (B, context_dim)

            spatial_contexts.append(ctx_t)
            spatial_weights_list.append(attn_w.squeeze(1))  # (B, 49)

        # (B, 5, context_dim) e (B, 5, 49)
        spat_stack = torch.stack(spatial_contexts,     dim=1)
        spat_w     = torch.stack(spatial_weights_list, dim=1)

        # ── Atenção temporal com query aprendida ──────────────────────────
        temp_keys = self.temp_key_proj(spat_stack)   # (B, 5, context_dim)
        temp_vals = self.temp_val_proj(spat_stack)   # (B, 5, context_dim)

        tq = self.temporal_query.unsqueeze(0).expand(B, -1, -1)  # (B,1,context_dim)
        temp_scores = torch.bmm(tq, temp_keys.transpose(1, 2)) / self.scale_temporal
        temp_w      = F.softmax(temp_scores, dim=-1)   # (B, 1, 5)

        temporal_ctx = torch.bmm(temp_w, temp_vals).squeeze(1)  # (B, context_dim)

        # Projeção final
        output = self.output_proj(temporal_ctx)  # (B, out_dim)

        return output, spat_w, temp_w.squeeze(1)  # (B,64), (B,5,49), (B,5)


# ── Teste do módulo ───────────────────────────────────────────────────────────
torch.manual_seed(SEED)
attn_v2 = SpatialAttentionV2().to(DEVICE)

dummy = torch.randint(0, 27, (4, 5, 7, 7)).float().to(DEVICE)
ctx, sw, tw = attn_v2(dummy)

n_params = sum(p.numel() for p in attn_v2.parameters())
print(f"✅ SpatialAttentionV2 criado!")
print(f"   Parâmetros: {n_params:,}")
print(f"   Output:            {ctx.shape}")
print(f"   Spatial weights:   {sw.shape}")
print(f"   Temporal weights:  {tw.shape}")
print()

# Verificar que weights não são uniformes no início
print(f"Pesos espaciais iniciais (antes do treino):")
print(f"   std por posição: {sw[0,0].std().item():.6f}  "
      f"(uniforme seria 0.0)")
print(f"   min: {sw[0,0].min().item():.6f}  "
      f"max: {sw[0,0].max().item():.6f}")
print()
print(f"Pesos temporais iniciais:")
print(f"   {tw[0].detach().cpu().numpy().round(4)}")
print(f"   std: {tw[0].std().item():.6f}")

In [ ]:
# ============================================================================
# GeoFM v3b — ARQUITETURA COMPLETA COM ATENÇÃO CORRIGIDA
# ============================================================================

class GeoFMv3b(nn.Module):
    """
    GeoFM v3b: como v3 mas com SpatialAttentionV2 (cross-attention).
    Interface idêntica ao GeoFMv3 — mesmos inputs e outputs.
    """

    def __init__(
        self,
        serie_dim   = 39,
        aux_dim     = 3,
        patch_years = 5,
        patch_n     = 7,
        temp_hidden = 256,
        temp_out    = 128,
        embed_dim   = 32,
        spatial_ctx = 64,
        spatial_out = 64,
        head_hidden = 64,
        dropout     = 0.3
    ):
        super().__init__()

        # Temporal encoder (série + aux) — idêntico ao v3
        temp_input_dim = serie_dim + aux_dim
        self.temporal_encoder = nn.Sequential(
            nn.Linear(temp_input_dim, temp_hidden),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(temp_hidden, temp_out),
            nn.ReLU(), nn.Dropout(dropout)
        )

        # Spatial attention CORRIGIDO
        self.spatial_attention = SpatialAttentionV2(
            patch_years = patch_years,
            patch_n     = patch_n,
            embed_dim   = embed_dim,
            context_dim = spatial_ctx,
            out_dim     = spatial_out,
            dropout     = dropout
        )

        # Prediction head
        combined_dim = temp_out + spatial_out
        self.head = nn.Sequential(
            nn.Linear(combined_dim, head_hidden),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
            nn.Sigmoid()
        )

    def forward(self, serie, patch, aux):
        temp_ctx             = self.temporal_encoder(torch.cat([serie, aux], dim=1))
        spat_ctx, spat_w, temp_w = self.spatial_attention(patch)
        pred = self.head(torch.cat([temp_ctx, spat_ctx], dim=1))
        return pred, spat_w, temp_w


# Verificação
torch.manual_seed(SEED)
model = GeoFMv3b().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

dummy_serie = torch.zeros(4, 39).to(DEVICE)
dummy_patch = torch.randint(0, 27, (4, 5, 7, 7)).float().to(DEVICE)
dummy_aux   = torch.zeros(4, 3).to(DEVICE)
pred, sw, tw = model(dummy_serie, dummy_patch, dummy_aux)

enc_p  = sum(p.numel() for p in model.temporal_encoder.parameters())
attn_p = sum(p.numel() for p in model.spatial_attention.parameters())
head_p = sum(p.numel() for p in model.head.parameters())

print(f"✅ GeoFMv3b criado!")
print(f"   Parâmetros totais: {total_params:,}")
print(f"     Temporal encoder:  {enc_p:,}")
print(f"     Spatial attention: {attn_p:,} (inclui queries aprendidas)")
print(f"     Prediction head:   {head_p:,}")
print()
print(f"   pred: {pred.shape}  |  spat_w: {sw.shape}  |  temp_w: {tw.shape}")
print()
print(f"   Comparação de parâmetros:")
print(f"     Baseline v2:             ~95,000")
print(f"     Multi-Head v2:           127,556")
print(f"     GeoFM v3 (colapsado):    ~mesmos")
print(f"     GeoFM v3b (corrigido):   {total_params:,}")

In [ ]:
# ============================================================================
# DATASET v3 — idêntico ao etapa6
# ============================================================================

import rasterio
from rasterio.windows import Window

DATA_DIR      = r"D:\Projetos\Cerrado\LULC"
PATTERN       = "brazil_coverage_{ano}_Cerrado.tif"
YEARS         = list(range(1985, 2025))
NODATA_IN     = 255
NODATA_OUT    = 0
PATCH_N       = 7
MAX_SERIE_LEN = len(YEARS) - 1
PATCH_YEARS   = 5

def _path(year):
    return os.path.join(DATA_DIR, PATTERN.format(ano=year))

def _ler_pixel(year, row, col):
    with rasterio.open(_path(year)) as ds:
        v = ds.read(1, window=Window(col, row, 1, 1), out_dtype="uint8")[0, 0]
    return int(v)

def _ler_patch(year, row, col, n=PATCH_N):
    half = n // 2
    with rasterio.open(_path(year)) as ds:
        H, W = ds.height, ds.width
        col0 = min(max(0, col - half), W - n)
        row0 = min(max(0, row - half), H - n)
        arr  = ds.read(1, window=Window(col0, row0, n, n), out_dtype="uint8")
    return np.where(arr == NODATA_IN, NODATA_OUT, arr).astype(np.uint8)


class LULCDatasetV3(Dataset):
    """Retorna serie (39), patch (5,7,7) e aux (3) separados."""

    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['label'].notna()].reset_index(drop=True)
        self.df['label'] = self.df['label'].astype(int)
        print(f"  Dataset: {len(self.df):,} samples")
        print(f"  Labels: {self.df['label'].value_counts().to_dict()}")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r   = self.df.iloc[idx]
        row = int(r['row']); col = int(r['col'])
        T   = int(r['T']);   label = int(r['label'])

        anos_serie = [y for y in YEARS if y < T]
        if len(anos_serie) > 0:
            serie_raw = np.array([_ler_pixel(y, row, col)
                                  for y in anos_serie], dtype=np.float32)
            serie_raw = np.where(serie_raw == NODATA_IN,
                                 NODATA_OUT, serie_raw).astype(np.float32)
        else:
            serie_raw = np.array([], dtype=np.float32)

        serie_len = len(serie_raw)
        serie     = np.zeros(MAX_SERIE_LEN, dtype=np.float32)
        if serie_len > 0:
            serie[MAX_SERIE_LEN - serie_len:] = serie_raw

        anos_patch = [y for y in YEARS if y < T][-PATCH_YEARS:]
        patch = np.zeros((PATCH_YEARS, PATCH_N, PATCH_N), dtype=np.float32)
        for i, y in enumerate(anos_patch):
            patch[PATCH_YEARS - len(anos_patch) + i] = _ler_patch(y, row, col)

        if serie_len > 0:
            has_21    = float(np.sum(serie_raw == 21)) / serie_len
            anos_past = sum(1 for v in reversed(serie_raw) if v == 15)
            cl_tm1    = float(serie_raw[-1])
        else:
            has_21 = anos_past = cl_tm1 = 0.0

        aux = np.array([has_21, float(anos_past), cl_tm1], dtype=np.float32)

        return (
            torch.tensor(serie,  dtype=torch.float32),
            torch.tensor(patch,  dtype=torch.float32),
            torch.tensor(aux,    dtype=torch.float32),
            torch.tensor([label], dtype=torch.float32)
        )


BASE_DIR  = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
SPLIT_DIR = BASE_DIR / "spatial_split"

print("Carregando splits...")
print("Train:"); train_dataset = LULCDatasetV3(SPLIT_DIR / "spatial_split_train.csv")
print("Val:");   val_dataset   = LULCDatasetV3(SPLIT_DIR / "spatial_split_val.csv")
print("Test:");  test_dataset  = LULCDatasetV3(SPLIT_DIR / "spatial_split_test.csv")

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\n✅ DataLoaders prontos! Batch size: {BATCH_SIZE}")

# Sanity check
serie, patch, aux, lbl = train_dataset[0]
assert serie.shape == (39,) and patch.shape == (5,7,7) and aux.shape == (3,)
print(f"   serie {serie.shape}  patch {patch.shape}  aux {aux.shape}  ✅")

In [ ]:
# ============================================================================
# LOSS COM REGULARIZAÇÃO DE ENTROPIA
# ============================================================================
# H(attn) = -sum(p * log(p))  → máxima quando uniforme
# Penalizar alta entropia força o modelo a concentrar atenção
# LAMBDA_ENTROPY: hiperparâmetro — começa pequeno, aumentar se necessário

LAMBDA_ENTROPY = 0.01   # peso da regularização

def entropy_loss(attn_weights):
    """
    Penalização de entropia nos pesos de atenção espacial.
    attn_weights: (batch, patch_years, n_positions)
    Retorna: escalar — entropia média (negativa para minimizar)
    """
    eps = 1e-8
    # Entropia por amostra e ano
    H = -(attn_weights * torch.log(attn_weights + eps)).sum(dim=-1)  # (B, 5)
    return H.mean()   # escalar


def train_epoch(model, loader, bce_criterion, optimizer, device,
                lambda_entropy=LAMBDA_ENTROPY):
    model.train()
    total_loss = total_bce = total_ent = 0
    correct = total = 0

    pbar = tqdm(loader, desc="Training")
    for serie, patch, aux, labels in pbar:
        serie  = serie.to(device)
        patch  = patch.to(device)
        aux    = aux.to(device)
        labels = labels.to(device)

        pred, spat_w, _ = model(serie, patch, aux)

        bce  = bce_criterion(pred, labels)
        ent  = entropy_loss(spat_w)
        # Penalizar entropia alta (uniforme) → subtrair entropia da loss
        # Minimizar (BCE - lambda*H) = maximizar H → concentrar atenção
        loss = bce - lambda_entropy * ent

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_bce  += bce.item()
        total_ent  += ent.item()
        pred_class  = (pred > 0.5).float()
        correct    += (pred_class == labels).sum().item()
        total      += labels.size(0)

        pbar.set_postfix({
            'bce':  f'{bce.item():.4f}',
            'ent':  f'{ent.item():.3f}',
            'acc':  f'{100*correct/total:.1f}%'
        })

    n = len(loader)
    return (total_loss/n, total_bce/n, total_ent/n, correct/total)


def evaluate(model, loader, bce_criterion, device, collect_weights=False):
    model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    all_spat_w, all_temp_w = [], []

    with torch.no_grad():
        for serie, patch, aux, labels in tqdm(loader, desc="Evaluating"):
            serie  = serie.to(device)
            patch  = patch.to(device)
            aux    = aux.to(device)
            labels = labels.to(device)

            pred, spat_w, temp_w = model(serie, patch, aux)
            loss = bce_criterion(pred, labels)

            total_loss += loss.item()
            pred_class  = (pred > 0.5).float()
            correct    += (pred_class == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            if collect_weights:
                all_spat_w.extend(spat_w.cpu().numpy())
                all_temp_w.extend(temp_w.cpu().numpy())

    preds_np  = np.array(all_preds).flatten()
    labels_np = np.array(all_labels).flatten()
    pc        = (preds_np > 0.5).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels_np, pc, average='binary')
    try:    auc = roc_auc_score(labels_np, preds_np)
    except: auc = 0.0

    metrics = {'loss': total_loss/len(loader), 'accuracy': correct/total,
               'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc}
    weights = (np.array(all_spat_w) if all_spat_w else None,
               np.array(all_temp_w) if all_temp_w else None)
    return metrics, weights


print(f"✅ Loss com regularização de entropia (lambda={LAMBDA_ENTROPY})")
print(f"   Entropia máxima (uniforme): {-np.log(1/49):.4f}")
print(f"   Entropia mínima (one-hot):  0.0")
print(f"   Penalização: loss = BCE - {LAMBDA_ENTROPY} × H(atenção)")

In [ ]:
# ============================================================================
# TREINO — GeoFM v3b
# ============================================================================

torch.manual_seed(SEED)
model = GeoFMv3b().to(DEVICE)

bce_criterion = nn.BCELoss()
optimizer     = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler     = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True)

N_EPOCHS       = 30
EARLY_STOP_PAT = 10

best_val_loss    = float('inf')
best_model_state = None
best_val_metrics = None
patience_counter = 0
history          = {'train_loss': [], 'train_bce': [], 'train_ent': [],
                    'train_acc': [], 'val_metrics': []}

print("=" * 70)
print("INICIANDO TREINO — GeoFM v3b")
print("=" * 70)
print(f"  Modelo:  GeoFMv3b ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Correção: cross-attention + query aprendida + entropy regularization")
print(f"  Lambda entropy: {LAMBDA_ENTROPY}")
print()

for epoch in range(N_EPOCHS):
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    print("-" * 70)

    train_loss, train_bce, train_ent, train_acc = train_epoch(
        model, train_loader, bce_criterion, optimizer, DEVICE)
    val_metrics, _ = evaluate(
        model, val_loader, bce_criterion, DEVICE, collect_weights=False)

    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_loss)
    history['train_bce'].append(train_bce)
    history['train_ent'].append(train_ent)
    history['train_acc'].append(train_acc)
    history['val_metrics'].append(val_metrics)

    print(f"\nTrain - Loss: {train_loss:.4f} "
          f"(BCE={train_bce:.4f}, Ent={train_ent:.4f}), "
          f"Acc: {train_acc:.4f}")
    print(f"Val   - Loss: {val_metrics['loss']:.4f}, "
          f"Acc: {val_metrics['accuracy']:.4f}, "
          f"F1: {val_metrics['f1']:.4f}, "
          f"AUC: {val_metrics['auc']:.4f}")

    # Monitorar entropia a cada epoch para verificar que não colapsou
    if epoch < 3 or (epoch + 1) % 5 == 0:
        print(f"   Entropia de treino: {train_ent:.4f} "
              f"({'OK - discriminando' if train_ent < 3.5 else 'ATENÇÃO - ainda uniforme'})")

    if val_metrics['loss'] < best_val_loss:
        best_val_loss    = val_metrics['loss']
        best_model_state = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        best_val_metrics = val_metrics.copy()
        patience_counter = 0
        print("  → Best model updated!")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PAT:
            print(f"\nEarly stopping após {epoch+1} epochs.")
            break

    print()

print("=" * 70)
print("TREINO CONCLUÍDO")
print("=" * 70)
print(f"  Epochs: {len(history['train_loss'])}")
print(f"  Melhor val loss: {best_val_loss:.4f}")
print(f"  Melhor val F1:   {best_val_metrics['f1']:.4f}")

In [ ]:
# ============================================================================
# AVALIAÇÃO FINAL + DIAGNÓSTICO DE ATENÇÃO
# ============================================================================

model.load_state_dict(best_model_state)
model.to(DEVICE)

print("=" * 70)
print("AVALIAÇÃO FINAL — TEST SET")
print("=" * 70)

test_metrics, (spat_w_np, temp_w_np) = evaluate(
    model, test_loader, bce_criterion, DEVICE, collect_weights=True)

val_acc      = best_val_metrics['accuracy']
test_acc     = test_metrics['accuracy']
val_test_gap = test_acc - val_acc

# Métricas
print(f"\n{'Métrica':<15} {'Validation':>12} {'Test':>12}")
print("-" * 42)
for k in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    print(f"{k:<15} {best_val_metrics[k]:>12.4f} {test_metrics[k]:>12.4f}")
print(f"\nVal-Test Gap: {val_test_gap*100:+.2f}pp")

# Diagnóstico de atenção — a questão central
print(f"\n{'='*70}")
print("DIAGNÓSTICO DE ATENÇÃO (Caminho B → funcionou?)")
print(f"{'='*70}")

flat_spat = spat_w_np.mean(axis=0).mean(axis=0)
spat_std  = flat_spat.std()
temp_std  = temp_w_np.std(axis=0).mean()
uniform   = 1.0 / 49

print(f"\nAtenção espacial:")
print(f"  Std por posição (média): {spat_std:.6f}  "
      f"({'✅ NÃO uniforme' if spat_std > 0.002 else '⚠️  Ainda uniforme'})")
print(f"  Posição mais atendida:   {flat_spat.argmax()} "
      f"→ [{flat_spat.argmax()//7}, {flat_spat.argmax()%7}] "
      f"(centro=[3,3]=24)")
print(f"  Max/uniforme ratio:      {flat_spat.max()/uniform:.2f}x")

print(f"\nAtenção temporal:")
mean_temp = temp_w_np.mean(axis=0)
year_labels = ['T-5', 'T-4', 'T-3', 'T-2', 'T-1']
for lbl, w in zip(year_labels, mean_temp):
    bar = '█' * int(w * 80)
    print(f"  {lbl}: {w:.4f}  {bar}")
print(f"  Std entre amostras: {temp_std:.6f}  "
      f"({'✅ discriminativo' if temp_std > 0.01 else '⚠️  Ainda uniforme'})")

# Tabela comparativa
print(f"\n{'='*70}")
print("COMPARAÇÃO COMPLETA")
print(f"{'='*70}")
print(f"{'Modelo':<38} {'Acc':>7} {'F1':>7} {'AUC':>7} {'Gap':>8}")
print("-" * 72)
for name, r in REFS.items():
    print(f"{name:<38} {r['acc']:>7.4f} {r['f1']:>7.4f} "
          f"{r['auc']:>7.4f} {r['gap']*100:>+7.1f}pp")
print(f"{'GeoFM v3b (atenção corrigida)':<38} "
      f"{test_acc:>7.4f} {test_metrics['f1']:>7.4f} "
      f"{test_metrics['auc']:>7.4f} {val_test_gap*100:>+7.1f}pp")

In [ ]:
# ============================================================================
# CONGELAR GeoFM v3b
# ============================================================================

BASE_DIR   = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
FREEZE_DIR = BASE_DIR / "geofmv3b_frozen"
FREEZE_DIR.mkdir(exist_ok=True, parents=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

checkpoint = {
    'model_state_dict': best_model_state,
    'model_class':      'GeoFMv3b',
    'config': {
        'serie_dim': 39, 'aux_dim': 3, 'patch_years': 5, 'patch_n': 7,
        'temp_hidden': 256, 'temp_out': 128,
        'embed_dim': 32, 'spatial_ctx': 64, 'spatial_out': 64,
        'head_hidden': 64, 'dropout': 0.3
    },
    'metrics': {
        'validation':   {k: float(v) for k, v in best_val_metrics.items()},
        'test':         {k: float(v) for k, v in test_metrics.items()},
        'val_test_gap': float(val_test_gap)
    },
    'training': {
        'n_epochs': len(history['train_loss']),
        'batch_size': BATCH_SIZE, 'lr': 0.001,
        'lambda_entropy': LAMBDA_ENTROPY,
        'optimizer': 'Adam', 'criterion': 'BCE + entropy_reg',
        'grad_clip': 1.0, 'seed': SEED
    },
    'fixes_vs_v3': [
        'Cross-attention with learned spatial query (nn.Parameter)',
        'Separate key_proj and val_proj networks (no circular dependency)',
        'Position embedding per grid cell (2D spatial prior)',
        'Learned temporal query (nn.Parameter)',
        f'Entropy regularization (lambda={LAMBDA_ENTROPY})'
    ]
}

pth_path = FREEZE_DIR / f"geofmv3b_frozen_{timestamp}.pth"
torch.save(checkpoint, pth_path)
print(f"✅ Modelo: {pth_path.name} ({pth_path.stat().st_size/(1024**2):.2f} MB)")

if spat_w_np is not None:
    np.save(FREEZE_DIR / f"spatial_weights_{timestamp}.npy",  spat_w_np)
    np.save(FREEZE_DIR / f"temporal_weights_{timestamp}.npy", temp_w_np)
    print(f"✅ Attention weights: spatial {spat_w_np.shape}, "
          f"temporal {temp_w_np.shape}")

results = {
    'model_info':  {'name': 'GeoFMv3b', 'frozen_date': timestamp},
    'performance': checkpoint['metrics'],
    'attention_diagnostics': {
        'spatial_std':    float(flat_spat.std()),
        'spatial_uniform': bool(flat_spat.std() < 0.002),
        'temporal_std':   float(temp_std),
        'temporal_uniform': bool(temp_std < 0.01),
        'top_position':   int(flat_spat.argmax()),
        'top_position_rc': [int(flat_spat.argmax()//7),
                            int(flat_spat.argmax()%7)]
    },
    'fixes_vs_v3': checkpoint['fixes_vs_v3']
}
json_path = FREEZE_DIR / f"results_{timestamp}.json"
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"✅ JSON: {json_path.name}")

print(f"\n{'='*70}")
print(f"GeoFM v3b CONGELADO — {FREEZE_DIR}")
print(f"{'='*70}")